In [ ]:
import os
import glob
import kagglehub

# Auto-detect runtime environment (Local vs Azure Databricks)
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
STORAGE_ACCOUNT = spark.conf.get("spark.storage.account.name", ".") if IS_DATABRICKS else ""
BASE_DATA_PATH = f"abfss://raw-data@{STORAGE_ACCOUNT}.dfs.core.windows.net" if IS_DATABRICKS else "../data"

# Download dataset via KaggleHub (works both locally and inside Databricks compute nodes)
output_dir = "/dbfs/tmp/raw" if IS_DATABRICKS else "../data/raw"
os.makedirs(output_dir, exist_ok=True)

download_path = kagglehub.dataset_download(
    "mmumairkhattak/e-commerce-orders-dataset-2026-scra",
    output_dir=output_dir
)
csv_files = glob.glob(os.path.join(output_dir, "*.csv"))
target_raw_csv = csv_files[1] if csv_files else f"{output_dir}/ecommerce_orders_dataset.csv"

print(f"Dataset ingested and ready at: {target_raw_csv}")

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder.appName("ECommerce_Data_Ingestion")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .getOrCreate()
)

print(f"Spark Version: {spark.version}")

In [ ]:
df_raw = spark.read.option("header", "true").csv(target_raw_csv)
print(f"Raw record count: {df_raw.count()}")
df_raw.show(5)

In [ ]:
df_raw.createOrReplaceTempView("raw_orders")

cleaned_orders_sql = """
SELECT 
    Order_ID,
    Customer_ID,
    Customer_Age,
    Customer_Gender,
    Country,
    City,
    COALESCE(Payment_Method, 'Unknown') AS Payment_Method,
    Order_Amount,
    TO_TIMESTAMP(Order_Date, 'yyyy-MM-dd') AS Order_Timestamp,
    CASE 
        WHEN Returned = 'Yes' THEN 1 
        ELSE 0 
    END AS Returned_Flag,
    CURRENT_TIMESTAMP() AS Ingested_At
FROM raw_orders
WHERE Order_Amount IS NOT NULL 
  AND Order_Amount > 0.0
"""

df_cleaned = spark.sql(cleaned_orders_sql)
df_cleaned.createOrReplaceTempView("cleaned_orders")

print(f"Cleaned row count: {df_cleaned.count()}")
df_cleaned.printSchema()
df_cleaned.show(5)

In [ ]:
curated_output_path = f"{BASE_DATA_PATH}/curated/cleaned_orders.parquet"

df_cleaned.write \
    .mode("overwrite") \
    .parquet(curated_output_path)

print(f"Cleaned dataset saved successfully to: {curated_output_path}")